In [1]:
import pandas as pd

df = pd.read_csv("../data_processed/all_stocks_daily.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['ticker', 'date'])

sector_df = pd.read_csv("../data_raw/Sector_data - Sheet1.csv")
sector_df.head(), sector_df.columns


(             COMPANY         sector                         Symbol
 0  ADANI ENTERPRISES  MISCELLANEOUS  ADANI ENTERPRISES: ADANIGREEN
 1  ADANI PORTS & SEZ  MISCELLANEOUS  ADANI PORTS & SEZ: ADANIPORTS
 2   APOLLO HOSPITALS  MISCELLANEOUS   APOLLO HOSPITALS: APOLLOHOSP
 3       ASIAN PAINTS         PAINTS       ASIAN PAINTS: ASIANPAINT
 4          AXIS BANK        BANKING            AXIS BANK: AXISBANK,
 Index(['COMPANY', 'sector', 'Symbol'], dtype='str'))

In [2]:
# Extract ticker from 'Symbol' column ("COMPANY: TICKER" -> "TICKER")
sector_df['ticker'] = sector_df['Symbol'].str.split(':').str[-1].str.strip()

# Keep only what we need
sector_df_clean = sector_df[['ticker', 'COMPANY', 'sector']].rename(
    columns={'COMPANY': 'company'}
)

sector_df_clean.head()


,ticker,company,sector
0,ADANIGREEN,ADANI ENTERPRISES,MISCELLANEOUS
1,ADANIPORTS,ADANI PORTS & SEZ,MISCELLANEOUS
2,APOLLOHOSP,APOLLO HOSPITALS,MISCELLANEOUS
3,ASIANPAINT,ASIAN PAINTS,PAINTS
4,AXISBANK,AXIS BANK,BANKING


In [3]:
sector_df_clean['ticker'] = sector_df_clean['ticker'].str.strip()
df['ticker'] = df['ticker'].str.strip()


#### Yearly return + sector join 

In [5]:
import pandas as pd

df = pd.read_csv("../data_processed/all_stocks_daily.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['ticker', 'date'])

first_last = df.groupby('ticker').agg(
    first_close=('close', 'first'),
    last_close=('close', 'last')
).reset_index()

first_last['yearly_return'] = (
    first_last['last_close'] - first_last['first_close']
) / first_last['first_close']

first_last.to_csv("../data_processed/yearly_return_per_ticker.csv", index=False)


In [6]:
sector_df = pd.read_csv("../data_raw/Sector_data - Sheet1.csv")
sector_df['ticker'] = sector_df['Symbol'].str.split(':').str[-1].str.strip()
sector_df_clean = sector_df[['ticker', 'COMPANY', 'sector']].rename(columns={'COMPANY': 'company'})

sector_df_clean['ticker'] = sector_df_clean['ticker'].str.strip()
first_last['ticker'] = first_last['ticker'].str.strip()

first_last_with_sector = first_last.merge(sector_df_clean, on='ticker', how='left')


In [7]:
first_last.to_csv("../data_processed/yearly_return_per_ticker.csv", index=False)


In [8]:
first_last = pd.read_csv("../data_processed/yearly_return_per_ticker.csv")

first_last_with_sector = first_last.merge(
    sector_df_clean,
    on='ticker',
    how='left'
)

first_last_with_sector.head()
first_last_with_sector[ first_last_with_sector['sector'].isna() ]


,ticker,first_close,last_close,yearly_return,company,sector
0,ADANIENT,2387.25,2228.00,-0.066709,NaN,NaN
9,BHARTIARTL,925.30,1569.30,0.695990,NaN,NaN
11,BRITANNIA,4495.45,4848.35,0.078502,NaN,NaN
41,TATACONSUM,861.20,945.20,0.097538,NaN,NaN


##### Sector wise performance table

In [9]:
sector_perf = (
    first_last_with_sector
    .dropna(subset=['sector'])
    .groupby('sector')['yearly_return']
    .mean()
    .reset_index()
    .sort_values('yearly_return', ascending=False)
)

sector_perf.head(20)


,sector,yearly_return
16,RETAILING,1.133054
4,DEFENCE,1.017601
15,POWER,0.601841
1,AUTOMOBILES,0.545265
11,MINING,0.418465
17,SOFTWARE,0.382760
3,CEMENT,0.369709
5,ENERGY,0.365648
12,MISCELLANEOUS,0.361031
0,ALUMINIUM,0.358683


##### Export

In [10]:
sector_perf.to_csv("../data_processed/sector_performance.csv", index=False)
first_last_with_sector.to_csv("../data_processed/yearly_return_with_sector.csv", index=False)
